In [ ]:
import sys
import os
import torch
import triton
# Add the parent directory to the Python path
sys.path.append(os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import matplotlib.patches as patches
import torch
import copy
from simu_PSF_polarMFM import *
from extract_experimental_psf import *
from tqdm import tqdm
import time
from torch.optim import SGD, Adam, AdamW
import torch.nn.functional as F

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
    print("Using GPU")
else:   
    device = torch.device('cpu')
    print("Using CPU")

In [ ]:
d = np.array([0.9,0.9,0.9])
d = np.array([np.mean(d)-0.365, np.mean(d), np.mean(d)+0.365])
QE = 0.92
EM = 200
sensitivity = 15.4

In [ ]:
def limit(x, lim, slope, upper=True):
    '''
    if upper:
       return torch.sum(torch.tensor(1/(1+torch.exp(-slope*(x-lim))), requires_grad=True, device=device))
    else:
        return torch.sum(torch.tensor(1/(1+torch.exp(slope*(x-lim))), requires_grad=True, device=device))
    '''
    if upper:
        return torch.sum(torch.exp((x-lim)*slope))
    else:
        return torch.sum(torch.exp(-1*(x-lim)*slope))
    
def loss_pos_torch(xp, yp, zp, rho, eta, delta, N_photons, data, second_plane, background, sigma, dim_simu, plot):
    M_ = compute_M(xp=xp, yp=yp, zp=zp, d=d_, x=xx, y=yy, th1=th1, phi=phi, Ex0=Ex0, Ex1=Ex1, Ex2=Ex2
                    , Ey0=Ey0, Ey1=Ey1, Ey2=Ey2, r=r, r_cut=r_cut, k=k_, f_o=f_o, phase_maskx=phase_mask, phase_masky=phase_mask, zernike_base=None, zernike_coefs_x=None, zernike_coefs_y=None,
                        second_plane=second_plane, polar_projections=polar_projections, lambd=lambd, f_tube=f_tube, 
                         device=device)
    dim_data = 6
    h = PSF(rho=rho, eta=eta, delta=delta, M=M_, N_photons=N_photons)[:,:,:,dim_simu-dim_data:dim_simu+dim_data+1,dim_simu-dim_data:dim_simu+dim_data+1]

    loss = torch.sum(torch.pow(torch.sum(torch.add(h+torch.reshape(background, (h.shape[0],3,2))[:, :, :, None, None], -data), dim=(2,)), 2))

    x_bound = limit(xp, 5*0.12, 100, upper=True) + limit(xp, -5*0.12, 100, upper=False)
    y_bound = limit(yp, 5*0.12, 100, upper=True) + limit(yp, -5*0.12, 100, upper=False)
    z_bound = limit(zp, 5., 100, upper=True) + limit(zp, 0, 100, upper=False)
    return (loss +x_bound+y_bound+z_bound).to(torch.float32)
#loss_pos = torch.compile(loss_pos_torch)
loss_pos = loss_pos_torch

def loss_angle_with_M_torch(rho, eta, delta, N_photons, x_fine, y_fine, z_fine, zernx, zerny, data, background, sigma, dim_simu):
    dim_data = 6
    M_ = compute_M(xp=x_fine, yp=y_fine, zp=z_fine, d=d_, x=xx, y=yy, th1=th1, phi=phi, Ex0=Ex0, Ex1=Ex1, Ex2=Ex2
                    , Ey0=Ey0, Ey1=Ey1, Ey2=Ey2, r=r, r_cut=r_cut, k=k_, f_o=f_o, phase_masky=phase_mask, phase_maskx=phase_mask, zernike_base=None, zernike_coefs_x=None, zernike_coefs_y=None,
                        second_plane=second_plane, polar_projections=polar_projections, lambd=lambd, f_tube=f_tube, device=device)
    h = PSF(rho=rho, eta=eta, delta=delta, M=M_, N_photons=N_photons)[:,:,:,dim_simu-dim_data:dim_simu+dim_data+1,dim_simu-dim_data:dim_simu+dim_data+1]
    
    loss = torch.sum(torch.add(h, -(data+sigma**2)*torch.log(h+torch.reshape(background, (h.shape[0],3,2))[:, :, :, None, None]+sigma**2)))
    delta_bound = limit(delta, 180, 100, upper=True) + limit(delta, 1, 100, upper=False)
    #rho_bound = limit(rho, 183, 100, upper=True) + limit(rho, -3, 100, upper=False)
    #eta_bound = limit(eta, 183, 100, upper=True) + limit(eta, -3, 100, upper=False)
    return (loss + 1000.*(delta_bound)).to(torch.float32) #+ N_bound #+ 100000*torch.sum(h**2)
#loss_angle_with_M = torch.compile(loss_angle_with_M_torch)
loss_angle_with_M = loss_angle_with_M_torch

def score_eval(M_, rho, eta, delta, N_photons, data, background, sigma, dim_simu):
    dim_data = 6
    h = PSF(rho=rho, eta=eta, delta=delta, M=M_, N_photons=N_photons)[:,:,:,dim_simu-dim_data:dim_simu+dim_data+1,dim_simu-dim_data:dim_simu+dim_data+1]
    score = torch.sum(torch.add(h, -(data+sigma**2)*torch.log(h+background+sigma**2)), dim=(1,2,3,4))
    return score.numpy() 

In [ ]:
J_dichroic = np.array([[0.7838338      ,               -0.25981125 + 1j*  -0.48329058],[
      -0.4230177 + 1j*  0.27765664  ,   -0.7788276 + 1j*  -0.28660256
]]) # this one is for the abstract
J_dichroic = np.array([[1.      ,               0.],[
      0. ,   1.
]])

In [ ]:
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [ ]:
npsf = 100

In [ ]:
eta_list = np.linspace(0.1, 179.9, 20)#.astype(int)
delta_list = np.linspace(0.1, 179.9, 20)#.astype(int)

In [ ]:
for eta__ in eta_list:
    for delta__ in delta_list:

        N=torch.tensor(80, device=device, requires_grad=False)
        l_pixel=torch.tensor(16, device=device, requires_grad=False)
        NA=torch.tensor(1.4, device=device, requires_grad=False)
        mag=torch.tensor(100, device=device, requires_grad=False)
        lambd=torch.tensor(638, device=device, requires_grad=False)
        f_tube=torch.tensor(200, device=device, requires_grad=False)
        MAG=torch.tensor(200/150, device=device, requires_grad=False)

        # microscope parameters
        d_ = -torch.tensor([d[1] for k in range(npsf)], requires_grad=False, device=device)
        second_plane = torch.tensor([d[1]-d[0], 0, d[1]-d[2]], device=device, requires_grad=False)
        polar_projections = np.array([0, 45, 0])
        
        SAF = False
    
        if SAF:
            xx, yy, th1, phi, [Ex0, Ex1, Ex2], [Ey0, Ey1, Ey2], r, r_cut, r_cut_saf, k_, f_o, costh2 = vectorial_BFP_perfect_focus(N, NA=NA, mag=mag, lambd=lambd, f_tube=f_tube, SAF=SAF, device=device, J_dichroic=J_dichroic)
        else:
            costh2=None
            xx, yy, th1, phi, [Ex0, Ex1, Ex2], [Ey0, Ey1, Ey2], r, r_cut, k_, f_o = vectorial_BFP_perfect_focus(N, NA=NA, mag=mag, lambd=lambd, f_tube=f_tube, SAF=SAF, device=device, J_dichroic=J_dichroic)
    
        u, v, Npadding = padding(r, r_cut, k_, f_o,  N=N, l_pixel=l_pixel, NA=NA, mag=mag, lambd=lambd, 
                  f_tube=f_tube, MAG=MAG, device=device)
    
        phase_mask = torch.stack([torch.ones((N,N), device=device), torch.ones((N,N), device=device), torch.ones((N,N), device=device)])
        zernike_base = None
        
        xx = pad(xx, Npadding).to(torch.complex64).detach()
        yy = pad(yy, Npadding).to(torch.complex64).detach()
        th1 = pad(th1, Npadding).to(torch.complex64).detach()
        phi = pad(phi, Npadding).to(torch.complex64).detach()
        Ex0 = pad(Ex0, Npadding).to(torch.complex64).detach()
        Ex1 = pad(Ex1, Npadding).to(torch.complex64).detach()
        Ex2 = pad(Ex2, Npadding).to(torch.complex64).detach()
        Ey0 = pad(Ey0, Npadding).to(torch.complex64).detach()
        Ey1 = pad(Ey1, Npadding).to(torch.complex64).detach()
        Ey2 = pad(Ey2, Npadding).to(torch.complex64).detach()
        phase_mask = pad(phase_mask, Npadding).to(torch.complex64).detach()
        #zernike_base = pad(zernike_base, Npadding).to(torch.complex64).detach()
        if SAF:
            costh2 = pad(costh2, Npadding).to(torch.complex64).detach()
    
        x_true = torch.tensor(2*(np.random.rand(npsf)-0.5)*0.2).to(device)
        y_true = torch.tensor(2*(np.random.rand(npsf)-0.5)*0.2).to(device)
        z_true = torch.tensor(np.random.rand(npsf)*0.8+0.2).to(device)
    
        Mtrue = compute_M(xp=x_true, yp=y_true, zp=z_true, d=d_, x=xx, y=yy, th1=th1, phi=phi, Ex0=Ex0, Ex1=Ex1, Ex2=Ex2
                    , Ey0=Ey0, Ey1=Ey1, Ey2=Ey2, r=r, r_cut=r_cut, k=k_, f_o=f_o, phase_maskx=phase_mask, phase_masky=phase_mask, zernike_base=None,
                        second_plane=second_plane, polar_projections=polar_projections, lambd=lambd, f_tube=f_tube, device=device)
    
        rho_true = torch.tensor(np.random.rand(npsf)*180).to(device)#torch.tensor([170 for i in range(npsf)]).to(device)#
        eta_true = torch.tensor([eta__ for uu in range(npsf)]).to(device)
        delta_true = torch.tensor([delta__ for uu in range(npsf)]).to(device)
        N_true = torch.tensor(np.random.rand(npsf)*5000.+3000.).to(device)

        psf_ = PSF(rho=rho_true, eta=eta_true, delta=delta_true, M=Mtrue, N_photons=N_true).to(torch.float32)
        psf = noise(psf_, QE=QE, EM=EM, b=15., sigma_b=7., sigma_r=7., bias=15.)
        
        nx = psf.shape[-1]
        noisy_psf = psf[:,:,:,nx//2-6:nx//2+7,nx//2-6:nx//2+7].to(torch.float32) 
        nb=0
        maxi = np.max(noisy_psf[nb,:,:].flatten().cpu().detach().numpy())
        fig, ax = plt.subplots(3,2)
        ax[0,0].imshow(noisy_psf[nb,0,0].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
        ax[1,0].imshow(noisy_psf[nb,1,0].cpu().detach().numpy() , vmin=0., vmax=maxi, cmap='gray')
        ax[2,0].imshow(noisy_psf[nb,2,0].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
        ax[0,1].imshow(noisy_psf[nb,0,1].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
        ax[1,1].imshow(noisy_psf[nb,1,1].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
        ax[2,1].imshow(noisy_psf[nb,2,1].cpu().detach().numpy(), vmin=0., vmax=maxi, cmap='gray')
        plt.show()
        del(fig, ax)

        sigma = torch.std(noisy_psf.flatten())
        background = torch.mean(noisy_psf.flatten())
    
        # nb of photons by plane roughly evaluated
        Nstart_by_plane = copy.deepcopy(torch.sum(noisy_psf, axis=(2,3,4)) - background*len(noisy_psf[0,0].flatten()))
    
        NPSF = noisy_psf.shape[0]
    
        # strating parameters (could be a first evaluation with coarse algo)
        x_start = torch.tensor([0. for k in range(NPSF)], requires_grad=False, device=device)
        y_start = torch.tensor([0. for k in range(NPSF)], requires_grad=False, device=device)
        z_exp =  torch.tensor([0.7 for k in range(NPSF)], requires_grad=False, device=device) 
    
        # convert to tensor
        #noisy_psf = torch.tensor(np.array([single_psf[k] for k in range(len(x))]), device=device, dtype=torch.float32)
    
        # starting point, could do a rough estimation first
        rho_start = torch.tensor([90. for k in range(NPSF)], device=device).to(torch.float32)
        eta_start = torch.tensor([90. for k in range(NPSF)], requires_grad=False, device=device).to(torch.float32)
        delta_start = torch.tensor([100. for k in range(NPSF)], device=device).to(torch.float32)
    
        # gradient descent parameters
        Nstart = torch.tensor([5000. for g in range(NPSF)], device=device).to(torch.float32)
        #Nstart = copy.deepcopy(torch.sum(noisy_psf, dim=(1,2,3,4)) - background*len(noisy_psf[0].flatten()))
        
        Mtest = compute_M(xp=x_start, yp=y_start, zp=z_exp, d=d_, x=xx, y=yy, th1=th1, phi=phi, Ex0=Ex0, Ex1=Ex1, Ex2=Ex2
                        , Ey0=Ey0, Ey1=Ey1, Ey2=Ey2, r=r, r_cut=r_cut, k=k_, f_o=f_o, phase_maskx=phase_mask, phase_masky=phase_mask, zernike_base=None, zernike_coefs_x=None, zernike_coefs_y=None,
                            second_plane=second_plane, polar_projections=polar_projections, lambd=lambd, f_tube=f_tube, device=device)
        
        htest = PSF(rho=rho_start, eta=eta_start, delta=delta_start, M=Mtest, N_photons=Nstart).to(torch.float32)
        
        dim_simu = int(htest.shape[-1]//2)
    
        background_array = (background*torch.ones((NPSF,3,2)).to(device)).to(torch.float32)
        
        params = torch.cat((x_start, y_start, z_exp, Nstart/3000, background_array.flatten())).to(torch.float32)
        params.requires_grad=True
    
        angle_rd = torch.tensor([180. for k in range(NPSF)], requires_grad=False, device=device).to(torch.float32)
        optimizer = torch.optim.Adam([params], lr=0.05)
        num_epochs_max = 80
        loss_ = []
        z__ = []
        N__ = []
        x__ =[]
        for i in tqdm(range(num_epochs_max)):
            optimizer.zero_grad()  # Reset gradients
            loss = loss_pos(params[0:NPSF], params[NPSF:2*NPSF], params[2*NPSF:3*NPSF], 
                               angle_rd, angle_rd, angle_rd, params[3*NPSF:4*NPSF]*3000, 
                               noisy_psf, second_plane, params[4*NPSF:10*NPSF], sigma, dim_simu, plot=False)
            loss_.append(loss.cpu().detach().numpy())
            z__.append(params[2*NPSF:3*NPSF].cpu().detach().numpy())
            N__.append((params[3*NPSF:4*NPSF]*3000).cpu().detach().numpy())
            x__.append((params[0*NPSF:1*NPSF]).cpu().detach().numpy())
            loss.backward()
            optimizer.step()
        ax = plt.plot(loss_)
        #plt.ylim((np.min(np.array(loss_)), np.max(np.array(loss_))))
        plt.show()
        ax = plt.plot(z__)
        plt.show()
        ax = plt.plot(N__)
        plt.show()
        ax = plt.plot(x__)
        #plt.ylim((np.min(np.array(x__)), np.max(np.array(x__))))
        plt.show()
        del(ax, loss_, z__, N__)
    
        x_found = params[0:NPSF].detach()
        y_found = params[NPSF:2*NPSF].detach()
        z_found = params[2*NPSF:3*NPSF].detach()
        N_found = params[3*NPSF:4*NPSF].detach()*3000
        bacground_array_found = params[4*NPSF:10*NPSF].detach()
        del(params, loss)
        print('NPSF = ', NPSF)
    
    #########################################################################################################################
    
        zern_x = torch.tensor(np.zeros(3*15), device=device)
        zern_y = torch.tensor(np.zeros(3*15), device=device)
    
        params = torch.cat((rho_start, eta_start, delta_start, N_found))#, x_found*10, y_found*10, z_found))
        params.requires_grad=True
    
        # Use Stochastic Gradient Descent (SGD) to optimize params
        optimizer = torch.optim.Adam([params], lr=1.5)  # Learning rate = 0.01
    
        num_epochs_max = 200
        loss_ = []
        eta_ = []
        rho_ = []
        delta_ = []
        for i in tqdm(range(num_epochs_max)):
            optimizer.zero_grad()  # Reset gradients
            loss = loss_angle_with_M(params[:NPSF], params[1*NPSF:2*NPSF], params[2*NPSF:3*NPSF], params[3*NPSF:4*NPSF], x_found, y_found, z_found, zern_x, zern_y, noisy_psf, bacground_array_found, sigma, dim_simu)
            loss_.append(loss.cpu().detach().numpy())
            eta_.append((params[1*NPSF:2*NPSF]-eta_true).cpu().detach().numpy())
            rho_.append((params[0*NPSF:1*NPSF]-rho_true).cpu().detach().numpy())
            delta_.append((params[2*NPSF:3*NPSF]-delta_true).cpu().detach().numpy())
            loss.backward()  # Backpropagation
            optimizer.step()  # Update parameters
        fig, ax = plt.subplots(2,2)
        ax[0,0].plot(loss_) 
        #ax[0,0].set_ylim((np.min(np.array(loss_)), np.max(np.array(loss_))))
        ax[0,1].plot(rho_)
        ax[1,0].plot(np.array(eta_)[:,np.abs((params[0*NPSF:1*NPSF]-rho_true).cpu().detach().numpy())>90], c='b')
        ax[1,0].plot(np.array(eta_)[:,np.abs((params[0*NPSF:1*NPSF]-rho_true).cpu().detach().numpy())<90], c='r')
        ax[1,1].plot(delta_)
        plt.show()
        del(fig, ax)
    
        rho_found=params[0:NPSF].detach()%360
        eta_found=params[1*NPSF:2*NPSF].detach()%360
        delta_found=params[2*NPSF:3*NPSF].detach()
        
        del(params, loss, eta_, loss_)
       
        np.savez_compressed('\\\\NAS_LOCCO\\Amaury\\DATA\\data_simu\\these_4P_MFM\\simu_precision_6\\eta_'+str(eta__)+'_delta_'+str(delta__)+'.npz', 
                            x_true=x_true.detach().cpu().numpy(), y_true=y_true.detach().cpu().numpy(), z_true=z_true.detach().cpu().numpy(),
                            N_true=N_true.detach().cpu().numpy(), rho_true=rho_true.detach().cpu().numpy(), eta_true=eta_true.detach().cpu().numpy(),
                            delta_true=delta_true.detach().cpu().numpy(),
                           x_found=x_found.detach().cpu().numpy(), y_found=y_found.detach().cpu().numpy(), z_found=z_found.detach().cpu().numpy(),
                            N_found=N_found.detach().cpu().numpy(), rho_found=rho_found.detach().cpu().numpy(), eta_found=eta_found.detach().cpu().numpy(),
                            delta_found=delta_found.detach().cpu().numpy())


In [ ]:
folder = '\\\\NAS_LOCCO\\Amaury\\DATA\\data_simu\\these_4P_MFM\\simu_precision_6\\'
x_true = []
y_true = []
z_true = []
rho_true = []
eta_true = []
delta_true = []
N_true = []
x_found = []
y_found = []
z_found = []
rho_found = []
eta_found = []
delta_found = []
N_found = []
for filename in os.listdir(folder):
    data = np.load(rf"{folder}\{filename}")
    x_true.append(data['x_true'])
    y_true.append(data['y_true'])
    z_true.append(data['z_true'])
    rho_true.append(data['rho_true'])
    eta_true.append(data['eta_true'])
    delta_true.append(data['delta_true'])
    N_true.append(data['N_true'])
    x_found.append(data['x_found'])
    y_found.append(data['y_found'])
    z_found.append(data['z_found'])
    rho_found.append(data['rho_found'])
    eta_found.append(data['eta_found'])
    delta_found.append(data['delta_found'])
    N_found.append(data['N_found'])
x_true = np.array(x_true).reshape(20,20,100)
y_true = np.array(y_true).reshape(20,20,100)
z_true = np.array(z_true).reshape(20,20,100)
rho_true = np.array(rho_true).reshape(20,20,100)
eta_true = np.array(eta_true).reshape(20,20,100)
delta_true = np.array(delta_true).reshape(20,20,100)
N_true = np.array(N_true).reshape(20,20,100)
x_found = np.array(x_found).reshape(20,20,100)
y_found = np.array(y_found).reshape(20,20,100)
z_found = np.array(z_found).reshape(20,20,100)
rho_found = np.array(rho_found).reshape(20,20,100)
eta_found = np.array(eta_found).reshape(20,20,100)
delta_found = np.array(delta_found).reshape(20,20,100)
N_found = np.array(N_found).reshape(20,20,100)

In [ ]:
mask0 = (rho_found>180) | (rho_found<0)
rho_found[mask0] = rho_found[mask0]%180
eta_found[mask0] = 90-eta_found[mask0]

In [ ]:
deltax = x_found-x_true
deltay = y_found-y_true
deltaz = z_found - z_true
deltarho = rho_found - rho_true
deltaeta = eta_found-eta_true
deltadelta = delta_found-delta_true
deltaN = N_found-N_true

In [ ]:
mask1 = deltarho > 90
mask2 = deltaeta > 90
deltarho[mask1] = deltarho[mask1]-180
deltaeta[mask2] = deltaeta[mask2]-180

In [ ]:
mask1 = deltarho < -90
mask2 = deltaeta < -90
deltarho[mask1] = deltarho[mask1]+180
deltaeta[mask2] = deltaeta[mask2]+180

In [ ]:
from matplotlib.colors import PowerNorm, LogNorm
norm1 = PowerNorm(gamma=0.6, vmin=0, vmax=max(np.max(np.std(deltaeta, axis=2)), np.max(np.std(deltarho, axis=2))))

In [ ]:
plt.rcParams['figure.figsize'] = [15, 3]
fig, ax = plt.subplots(1,3)
mesh = ax[0].pcolormesh(eta_list, delta_list, np.std(deltarho, axis=2).T, norm=norm1)
ax[0].set_xlabel('$\\eta$ ($\\degree$)')
ax[0].set_ylabel('$\\delta$ ($\\degree$)')
cb = plt.colorbar(mesh)

mesh = ax[1].pcolormesh(eta_list, delta_list, np.std(deltaeta, axis=2).T, norm=norm1)
ax[1].set_xlabel('$\\eta$ ($\\degree$)')
ax[1].set_ylabel('$\\delta$ ($\\degree$)')
cb = plt.colorbar(mesh)

mesh = ax[2].pcolormesh(eta_list, delta_list, np.std(deltadelta, axis=2).T, norm=norm1)
ax[2].set_xlabel('$\\eta$ ($\\degree$)')
ax[2].set_ylabel('$\\delta$ ($\\degree$)')
cb = plt.colorbar(mesh)
plt.savefig("std_orientation.svg", format="svg")

In [ ]:
plt.rcParams['figure.figsize'] = [5, 3]
fig, ax = plt.subplots(1)

mesh = ax.pcolormesh(eta_list, delta_list, np.mean(deltadelta, axis=2).T, norm=norm1)
ax.set_xlabel('$\\eta$ ($\\degree$)')
ax.set_ylabel('$\\delta$ ($\\degree$)')
cb = plt.colorbar(mesh)
plt.savefig("bias_delta.svg", format="svg")

In [ ]:
norm2 = LogNorm(vmin=0.005, vmax=0.2)

plt.rcParams['figure.figsize'] = [15, 3]
fig, ax = plt.subplots(1,3)
mesh = ax[0].pcolormesh(eta_list, delta_list, np.mean(np.sqrt(deltax**2+deltay**2), axis=2).T, norm=norm2)
ax[0].set_xlabel('$\\eta$ ($\\degree$)')
ax[0].set_ylabel('$\\delta$ ($\\degree$)')
cb = plt.colorbar(mesh)

mesh = ax[1].pcolormesh(eta_list, delta_list, np.mean(np.abs(deltaz), axis=2).T, norm=norm2)
ax[1].set_xlabel('$\\eta$ ($\\degree$)')
ax[1].set_ylabel('$\\delta$ ($\\degree$)')
cb = plt.colorbar(mesh)

mesh = ax[2].pcolormesh(eta_list, delta_list,  np.mean(deltaN, axis=2).T)
ax[2].set_xlabel('$\\eta$ ($\\degree$)')
ax[2].set_ylabel('$\\delta$ ($\\degree$)')
cb = plt.colorbar(mesh)
plt.savefig("bias_loc.svg", format="svg")

In [ ]:
norm3 = LogNorm(vmin=0.005, vmax=0.1)
norm4 = LogNorm(vmin=0.01, vmax=0.3)

plt.rcParams['figure.figsize'] = [15, 3]
fig, ax = plt.subplots(1,3)
mesh = ax[0].pcolormesh(eta_list, delta_list, np.std(np.sqrt(deltax**2+deltay**2), axis=2).T, norm=norm3)
ax[0].set_xlabel('$\\eta$ ($\\degree$)')
ax[0].set_ylabel('$\\delta$ ($\\degree$)')
cb = plt.colorbar(mesh)

mesh = ax[1].pcolormesh(eta_list, delta_list, np.std(np.abs(deltaz), axis=2).T, norm=norm4)
ax[1].set_xlabel('$\\eta$ ($\\degree$)')
ax[1].set_ylabel('$\\delta$ ($\\degree$)')
cb = plt.colorbar(mesh)

mesh = ax[2].pcolormesh(eta_list, delta_list, np.std(deltaN, axis=2).T)
ax[2].set_xlabel('$\\eta$ ($\\degree$)')
ax[2].set_ylabel('$\\delta$ ($\\degree$)')
cb = plt.colorbar(mesh)
plt.savefig("std_loc.svg", format="svg")